# mjo_spectra_season

- Calculates seasonal spectra via segment averaging as defined by the US-CLIVAR MJO diagnostics website
- [NCL: mjo_spectra_season](https://www.ncl.ucar.edu/Document/Functions/Diagnostics/mjo_spectra_season.shtml)

[NCL Script](mjo_spectra_season.ncl)

- [NCL function: mjo_spectra_season](https://github.com/NCAR/ncl/blob/8f9e9476281cc6f6d9d12eaa78729c7003ca24b7/ni/src/examples/gsun/diagnostics_cam.ncl#L2257)

Input:
- X: three-dimensional variable (time, lat, lon)
- date: dates array as YYYYMMDD
- wy: array containing latitude weights associated with x
- opt

Returns:
- Bandwidth (float)
- Average of input variance (float)
- Average lag one day autocorrelation (float)
- Average spectra (1D array of floats)
- Frequency (1D array of floats)

In [55]:
import xarray as xr
import os
import numpy as np
import scipy

In [56]:
time_start = "1979-01-01"
time_end = "1981-12-31"

In [57]:
flut_data = xr.open_dataset(os.getcwd() + "/data/anomaly/QBOi.EXP1.AMIP.001.flut.day.anom.nc")
flut_data = flut_data.sel(time=slice(time_start, time_end))
flut_data

<xarray.Dataset> Size: 242MB
Dimensions:  (time: 1095, lat: 192, lon: 288)
Coordinates:
  * time     (time) object 9kB 1979-01-01 00:00:00 ... 1981-12-31 00:00:00
  * lat      (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.3 357.5 358.8
Data variables:
    date     (time) float64 9kB ...
    FLUT     (time, lat, lon) float32 242MB ...
Attributes:
    history:  Thu Mar  6 13:31:31 2025: ncatted -a cell_methods,FLUT,m,c,time...
    NCO:      netCDF Operators version 5.3.1 (Homepage = http://nco.sf.net, C...

In [122]:
def specx_anal(data, detrend_opt, smooth_opt, percent_taper):
    # https://ncar.github.io/geocat-applications/ncl/ncl_entries/spectral_analysis.html
    #  specx_anal(xts[iStrt:iLast], d, sm, pct)
    print("running specx_anal()")
    
    # Optionally detrends the series
    if detrend_opt == 0:
        # remove mean
        data_detrend = data - data.mean()
    if detrend_opt == 1:
        # remove mean and detrend
        data_detrend = data - data.mean()
        data_detrend = scipy.signal.detrend(data_detrend, type="constant")
    
    # Optionally tapers the series
    if percent_taper == 0:
        # percent tapered: (0.0 <= pct <= 1.0) 0.10 common
        # generate periodic window
        tukey_window = scipy.signal.windows.tukey(len(data_detrend),
                                                 alpha=percent_taper,
                                                 sym=False)
        data_tapered = data_detrend * tukey_window

    # Calculate the variance of the detrended/tapered series
    data_variance = data_tapered.var(dim="time")

    # Periodogram (FFT and Fourier Cofficents)
    freq_data, psd_data = scipy.signal.periodogram(data_tapered,
                                                   fs=1, # samples monthly
                                                   detrend=False)
    # Smooth the periodogram
    if smooth_opt == 0:
        # 0 means no smooth (pure periodgram)
        smoothed_psd = psd_data
    else:
        # TODO: check on smoothing constants when not 0
        k = 7 # smooth constant
        kernel = np.ones(7) # Daniel smoothing kernel
        kernel[0] = (0.5) #  "Modify" kernel by making the endpoints have half the weight of the interior points
        kernel[-1] = 0.5
        kernel = kernel / kernel.sum()
        smoothed_psd = scipy.signal.convolve(psd_data, kernel, mode="same") # sets output array as same length as the first input

    # Noramlizes the periodogram so the area under the curves matches the calculated variance
    df = freq_data[1] - freq_data[0] # frequency step
    ## Create array to adjust contributions of endpoints
    frac = np.ones_like(freq_data)
    frac[0] = 0.5
    frac[-1] = 0.5
    current_area = np.sum(smoothed_psd * df * frac) # calculate current area under the curve
    normalized_factor = data_variance / current_area # find fact to adjust this area
    normalized_psd = (smoothed_psd * normalized_factor.values) # apply normalization factor to the smoothed power spectrum

    spectra_degrees_of_freedom = normalized_psd
    return spectra_degrees_of_freedom

def mjo_spectra_season(data, date, lat_weights, season_name, opt=False):
    # Winter starts November 1 (180 days)
    # Summer starts May 1 (180 days)
    # Annual starts January 1 (365 days)

    if len(data) == 0:
        print("mjo_spectra: currently, missing data not allowed")

    dimx = data.shape
    dimwy = lat_weights.shape
    if dimx[1] != dimwy[0]:
        print("band_pass_area_time: sizes of y/lat dimension do not match")
        print(f"\twy = {dimwy[0]}")
        print(f"\tdimx(1) = nlat = {dimx[1]}")
        exit()
    ntime = dimx[0] # time steps

    xts = data.weighted(lat_weights).mean(dim=["lat", "lon"]) # wgt_areaave_Wrap(x, wy, 1., 0) for time

    ## TODO: add in opt from line 2289, currently ignores as opt=False
    # https://github.com/NCAR/ncl/blob/8f9e9476281cc6f6d9d12eaa78729c7003ca24b7/ni/src/examples/gsun/diagnostics_cam.ncl#L2289

    if season_name == "winter":
        # November 1
        mmStart = 11
        ddStart = 1 
        nDay = 180
    if season_name == "summer":
        # May 1
        mmStart = 5
        ddStart = 1
        nDay = 180
    if season_name == "annual":
        # January 1
        mmStart = 1
        ddStart = 1
        nDay = 365

    is_start_date_mask = (data["time.month"] == mmStart) & (data["time.day"] == ddStart)
    iSea = data.where(is_start_date_mask, drop=True) # places where it is start date
    iSea = iSea.time.values
    nSea = len(iSea)
    if season_name == "winter":
        nSea -= 1 # last season not complete

    # calculate spectrum via segment averaging
    # MJO Clivar says "no" to detrending/tapering
    # TODO: ignoring opt options
    d = 0     # detrending opt: 0 => remove mean 1 => remove mean and detrend
    sm = 0    # smooth periodogram: 0 means no smooth (pure periodgram)
    pct = 0.0 # percent tapered: (0.0 <= pct <= 1.0) 0.10 common

    spec = np.zeros([int(nDay/2)])
    z1 = 0.0
    xtsVar = 0.0

    # segemnt averaging
    import cftime
    for ns in range(nSea):
        iStrt = iSea[ns]
        num_as_date = cftime.date2num(iStrt, units="days since 2000-01-01 00:00:00", calendar="noleap")
        last_date = num_as_date + (nDay - 1)
        iLast = cftime.num2date(last_date, units="days since 2000-01-01 00:00:00", calendar="noleap")
        print(iStrt)
        print(iLast)
        input_data = xts.sel(time=slice(iStrt,iLast))
        sdof  = specx_anal(data=input_data,
                        detrend_opt=d,
                        smooth_opt=sm, 
                        percent_taper=pct)
        spect += sdof
        break

In [123]:
input_data = flut_data.FLUT
dates = flut_data.date
wy = np.cos(np.deg2rad(flut_data.lat))
season = "summer"

mjo_spectra_season(data=input_data, date=flut_data.date, lat_weights=wy, season_name=season)

1979-05-01 00:00:00
1979-10-27 00:00:00
running specx_anal()
